# AM01 Colab Master Notebook

Notebook operativo per eseguire il progetto AM01 su Google Colab. Il notebook non reimplementa la pipeline: richiama gli script del repository, salva risultati su Google Drive e produce gli artefatti necessari per analisi, report e presentazione.

Flusso consigliato:
1. setup ambiente;
2. test/smoke sintetico;
3. audit dataset reale;
4. preprocessing leakage-safe;
5. training baseline e AE/AAE;
6. figure e tabelle;
7. ablation opzionale.

## 0. Parametri

Prima di eseguire tutto, imposta `REPO_URL` se vuoi clonare il repository da GitHub. Il dataset reale deve essere disponibile in Drive con la struttura `KukaColumnNames.npy`, `KukaNormal.npy`, `KukaSlow.npy`.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Bernuz2003/AML_anomaliy_detection.git"
PROJECT_DIR = Path("/content/am01-kuka-aae-anomaly-detection")
DRIVE_ROOT = Path("/content/drive/MyDrive/AM01")
DATA_DIR = DRIVE_ROOT / "data" / "KukaVelocityDataset"
RESULTS_DIR = DRIVE_ROOT / "results"
PROCESSED_DIR = DRIVE_ROOT / "data" / "processed" / "kuka_default"

RUN_DEEP_MODELS = True
RUN_CONV1D = True
RUN_ABLATION = True

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

DATA_DIR: /content/drive/MyDrive/AM01/data/KukaVelocityDataset
RESULTS_DIR: /content/drive/MyDrive/AM01/results


## 1. Mount Drive, clone/install e verifica ambiente

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [3]:
import os
import subprocess

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError("PROJECT_DIR non esiste. Imposta REPO_URL o carica/clona il repo in /content.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

def sh(cmd: str) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

Working directory: /content/am01-kuka-aae-anomaly-detection


In [4]:
!python --version
!pip install -q -r requirements.txt
!python -m compileall -q src scripts tests

Python 3.12.13


## 2. Test e smoke test sintetico

Questa sezione controlla che il codice funzioni prima di usare il dataset reale.

In [5]:
!pytest -q
!python scripts/run_synthetic_smoke.py

..............                                                           [100%]
python3: can't open file '/content/am01-kuka-aae-anomaly-detection/scripts/run_synthetic_smoke.py': [Errno 2] No such file or directory


## 3. Verifica dataset reale

In [6]:
required = ["KukaColumnNames.npy", "KukaNormal.npy", "KukaSlow.npy"]
missing = [name for name in required if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"File mancanti in {DATA_DIR}: {missing}")
print("Dataset trovato:")
for name in required:
    path = DATA_DIR / name
    print(name, path.stat().st_size, "bytes")

Dataset trovato:
KukaColumnNames.npy 13004 bytes
KukaNormal.npy 160849024 bytes
KukaSlow.npy 28910576 bytes


## 4. Data audit ed esempi di segnali

In [7]:
!python scripts/audit_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{RESULTS_DIR / 'data_audit'}"
!python scripts/plot_data_examples.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{RESULTS_DIR / 'figures' / 'data_examples'}"

Data audit completed.
Rows: 275330
Runs: 2643
Features: 85
Anomalous rows: 41538
Outputs written to /content/drive/MyDrive/AM01/results/data_audit
Signal figures written to /content/drive/MyDrive/AM01/results/figures/data_examples


In [8]:
import pandas as pd

summary_path = RESULTS_DIR / "data_audit" / "dataset_summary.csv"
summary = pd.read_csv(summary_path)
display(summary[["run_id", "n_rows", "run_label", "anomaly_fraction", "source_file"]].head())
print("Runs:", len(summary))
print(summary["run_label"].value_counts(dropna=False))

,run_id,n_rows,run_label,anomaly_fraction,source_file
0,normal_seg_0000,142,normal,0.0,normal
1,normal_seg_0001,108,normal,0.0,normal
2,normal_seg_0002,110,normal,0.0,normal
3,normal_seg_0003,110,normal,0.0,normal
4,normal_seg_0004,111,normal,0.0,normal


Runs: 2643
run_label
normal       2340
anomalous     303
Name: count, dtype: int64


## 5. Preprocessing leakage-safe

Crea finestre train/validation/test, scaler e summary. Lo scaler viene fitatto solo sui normali di training.

In [9]:
!python scripts/prepare_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{PROCESSED_DIR}"
!ls -lh "{PROCESSED_DIR}"

Data preparation completed.
{'rows': {'train': 165495, 'val': 54623, 'test': 55212}, 'runs': {'train': 1586, 'val': 529, 'test': 528}, 'windows': {'train': 4725, 'val': 1537, 'test': 1564, 'train_normal': 3801}, 'feature_cols': ['machine_nameKuka Robot_apparent_power', 'machine_nameKuka Robot_current', 'machine_nameKuka Robot_frequency', 'machine_nameKuka Robot_phase_angle', 'machine_nameKuka Robot_power', 'machine_nameKuka Robot_power_factor', 'machine_nameKuka Robot_reactive_power', 'machine_nameKuka Robot_voltage', 'sensor_id1_AccX', 'sensor_id1_AccY', 'sensor_id1_AccZ', 'sensor_id1_GyroX', 'sensor_id1_GyroY', 'sensor_id1_GyroZ', 'sensor_id1_q1', 'sensor_id1_q2', 'sensor_id1_q3', 'sensor_id1_q4', 'sensor_id1_temp', 'sensor_id2_AccX', 'sensor_id2_AccY', 'sensor_id2_AccZ', 'sensor_id2_GyroX', 'sensor_id2_GyroY', 'sensor_id2_GyroZ', 'sensor_id2_q1', 'sensor_id2_q2', 'sensor_id2_q3', 'sensor_id2_q4', 'sensor_id2_temp', 'sensor_id3_AccX', 'sensor_id3_AccY', 'sensor_id3_AccZ', 'sensor_id3

## 6. Run principale: PCA, Isolation Forest, AE, AAE e Conv1D-AE

In [10]:
MAIN_RUNS_DIR = RESULTS_DIR / "runs" / "main"
MAIN_RUNS_DIR.mkdir(parents=True, exist_ok=True)

pca_dir = MAIN_RUNS_DIR / "pca"
iforest_dir = MAIN_RUNS_DIR / "isolation_forest"
ae_dir = MAIN_RUNS_DIR / "ae_mlp"
aae_dir = MAIN_RUNS_DIR / "aae_mlp"
conv_dir = MAIN_RUNS_DIR / "ae_conv1d"

sh(f'python scripts/train.py --config configs/pca.yaml --data "{DATA_DIR}" --output "{pca_dir}"')
sh(f'python scripts/train.py --config configs/isolation_forest.yaml --data "{DATA_DIR}" --output "{iforest_dir}"')

if RUN_DEEP_MODELS:
    sh(f'python scripts/train.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{ae_dir}"')
    sh(f'python scripts/train.py --config configs/aae_mlp.yaml --data "{DATA_DIR}" --output "{aae_dir}"')
if RUN_CONV1D:
    sh(f'python scripts/train.py --config configs/ae_conv1d.yaml --data "{DATA_DIR}" --output "{conv_dir}"')


$ python scripts/train.py --config configs/pca.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/main/pca"

$ python scripts/train.py --config configs/isolation_forest.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/main/isolation_forest"

$ python scripts/train.py --config configs/ae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/main/ae_mlp"

$ python scripts/train.py --config configs/aae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/main/aae_mlp"

$ python scripts/train.py --config configs/ae_conv1d.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/main/ae_conv1d"


## 7. Tabelle metriche e figure

In [11]:
import json

rows = []
for metrics_path in sorted(MAIN_RUNS_DIR.glob("*/metrics.json")):
    with metrics_path.open() as f:
        metrics = json.load(f)
    row = {"run": metrics_path.parent.name, "threshold": metrics.get("threshold")}
    for key, value in metrics.get("test_metrics", {}).items():
        row[f"test_{key}"] = value
    rows.append(row)

metrics_table = pd.DataFrame(rows).sort_values("run")
tables_dir = RESULTS_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
metrics_csv = tables_dir / "main_metrics.csv"
metrics_table.to_csv(metrics_csv, index=False)
display(metrics_table)
print("Saved:", metrics_csv)

,run,threshold,test_threshold,test_precision,test_recall,test_f1,test_balanced_accuracy,test_tn,test_fp,test_fn,...,test_roc_auc,test_pr_auc,test_event_recall,test_event_precision,test_true_events,test_predicted_events,test_false_predicted_events,test_false_alarms_per_run,test_mean_false_alarm_duration_windows,test_mean_detection_delay
0,aae_mlp,0.330609,0.330609,0.440650,0.846875,0.579679,0.785174,900.0,344.0,49.0,...,0.826580,0.443691,0.883333,0.235043,60.0,234.0,179.0,0.339658,1.921788,3.320755
1,ae_conv1d,0.485910,0.485910,0.539568,0.703125,0.610583,0.774392,1052.0,192.0,95.0,...,0.828104,0.457230,0.683333,0.308824,60.0,136.0,94.0,0.178368,2.042553,4.292683
2,ae_mlp,0.297384,0.297384,0.468333,0.878125,0.610870,0.810847,925.0,319.0,39.0,...,0.848136,0.457011,0.950000,0.263889,60.0,216.0,159.0,0.301708,2.006289,3.368421
3,isolation_forest,-0.039149,-0.039149,0.616253,0.853125,0.715596,0.858235,1074.0,170.0,47.0,...,0.936962,0.719646,0.900000,0.435714,60.0,140.0,79.0,0.149905,2.151899,1.777778
4,pca,0.074957,0.074957,0.303390,0.559375,0.393407,0.614495,833.0,411.0,141.0,...,0.688281,0.287317,0.900000,0.226974,60.0,304.0,235.0,0.445920,1.748936,7.407407


Saved: /content/drive/MyDrive/AM01/results/tables/main_metrics.csv


In [12]:
for run_dir in sorted(MAIN_RUNS_DIR.iterdir()):
    if run_dir.is_dir() and (run_dir / "scores_test.csv").exists():
        sh(f'python scripts/plot_results.py --run-dir "{run_dir}"')

for name in ["ae_mlp", "aae_mlp", "ae_conv1d"]:
    run_dir = MAIN_RUNS_DIR / name
    if run_dir.exists() and (run_dir / "model.pt").exists():
        sh(f'python scripts/plot_model_diagnostics.py --run-dir "{run_dir}" --split test')


$ python scripts/plot_results.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/aae_mlp"

$ python scripts/plot_results.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/ae_conv1d"

$ python scripts/plot_results.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/ae_mlp"

$ python scripts/plot_results.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/isolation_forest"

$ python scripts/plot_results.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/pca"

$ python scripts/plot_model_diagnostics.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/ae_mlp" --split test

$ python scripts/plot_model_diagnostics.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/aae_mlp" --split test

$ python scripts/plot_model_diagnostics.py --run-dir "/content/drive/MyDrive/AM01/results/runs/main/ae_conv1d" --split test


## 8. Ablation opzionale

Da eseguire solo dopo aver verificato il run principale. Produce una tabella `experiment_summary.csv`.

In [13]:
if RUN_ABLATION:
    ABLATION_DIR = RESULTS_DIR / "runs" / "ablation"
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{ABLATION_DIR}" '
        f'--seeds 0 1 2 '
        f'--window-lengths 32 64 128 '
        f'--latent-dims 8 16 32 '
        f'--lambda-advs 0.01 0.1 1.0'
    )
else:
    print("Ablation disattivata. Imposta RUN_ABLATION=True nella sezione parametri.")


$ python scripts/run_experiments.py --configs configs/ae_mlp.yaml configs/aae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/runs/ablation" --seeds 0 1 2 --window-lengths 32 64 128 --latent-dims 8 16 32 --lambda-advs 0.01 0.1 1.0


## 9. Manifest risultati

Usa questa lista per scaricare o copiare gli artefatti da Drive nel report.

In [14]:
!find "{RESULTS_DIR}" -maxdepth 5 -type f | sort | sed -n '1,240p'

/content/drive/MyDrive/AM01/results/analysis/auto_insights.md
/content/drive/MyDrive/AM01/results/analysis/figures/aae_adversarial_losses.png
/content/drive/MyDrive/AM01/results/analysis/figures/aae_minus_ae_deltas.png
/content/drive/MyDrive/AM01/results/analysis/figures/bar_test_f1.png
/content/drive/MyDrive/AM01/results/analysis/figures/bar_test_false_alarms_per_run.png
/content/drive/MyDrive/AM01/results/analysis/figures/bar_test_pr_auc.png
/content/drive/MyDrive/AM01/results/analysis/figures/bar_test_roc_auc.png
/content/drive/MyDrive/AM01/results/analysis/figures/dataset_run_length_distribution.png
/content/drive/MyDrive/AM01/results/analysis/figures/dataset_runs_by_label.png
/content/drive/MyDrive/AM01/results/analysis/figures/metric_profile_by_model.png
/content/drive/MyDrive/AM01/results/analysis/figures/score_correlation_heatmap.png
/content/drive/MyDrive/AM01/results/analysis/figures/test_pr_curves_combined.png
/content/drive/MyDrive/AM01/results/analysis/figures/test_roc_cur